In [ ]:
# ════════════════════════════════════════════════════════════════
# GRPO only — starting from 0.72 SFT adapter
# ════════════════════════════════════════════════════════════════

USE_GRPO = True
QUICK_TEST = False
SMOKE_TEST = False

BASE_MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('FULL GRPO TRAINING — bit_manipulation only, from 0.72 SFT adapter')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: INSTALL vLLM (pre-built wheel for sm_120 + torch 2.12.0.dev)
# ════════════════════════════════════════════════════════════════
import os, sys, subprocess, glob, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Find vLLM wheel — Kaggle strips '+' from filenames, need to fix
vllm_wheels = glob.glob('/kaggle/input/**/vllm*.whl', recursive=True)
assert vllm_wheels, 'vLLM wheel not found'

whl = vllm_wheels[0]
fixed_whl = whl
if '+' not in os.path.basename(whl) and 'cu128' in os.path.basename(whl):
    fixed_name = os.path.basename(whl).replace('0.18.0cu128', '0.18.0+cu128')
    fixed_whl = os.path.join('/tmp', fixed_name)
    shutil.copy2(whl, fixed_whl)

# Install vllm WITH deps from offline packages
find_links = []
for d in glob.glob('/kaggle/input/**/offline_packages', recursive=True):
    find_links.extend(['--find-links', d])
for d in glob.glob('/kaggle/input/**/packages', recursive=True):
    find_links.extend(['--find-links', d])

# First install missing deps that vllm needs
deps = ['msgspec', 'partial-json-parser', 'compressed-tensors', 'depyf',
        'cloudpickle', 'prometheus-client', 'uvloop', 'fastapi', 'uvicorn',
        'pynvml', 'blake3', 'lm-format-enforcer', 'outlines']
for dep in deps:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-index', dep] + find_links,
        capture_output=True, text=True,
    )

# Then install vllm wheel (no deps — we just installed them)
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', fixed_whl],
    capture_output=True, text=True,
)
assert r.returncode == 0, f'vLLM install failed: {r.stderr[-500:]}'

import vllm
print(f'✅ vLLM {vllm.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3: PACKAGE INSTALL (trl, mamba_ssm)
# ════════════════════════════════════════════════════════════════
import glob, importlib.util, subprocess, sys, types

def sh(cmd, check=True):
    print('+', cmd)
    r = subprocess.run(cmd, shell=True, check=check, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  STDERR: {r.stderr[-300:]}')
    return r

def find_spec(name):
    return importlib.util.find_spec(name) is not None

def recursive_wheels(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))

def pick_best(wheels):
    py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    py_only = [w for w in wheels if py_tag in w]
    if py_only: return py_only[-1]
    return wheels[-1] if wheels else None

# Collect offline package dirs
find_links = []
for d in glob.glob('/kaggle/input/**/offline_packages', recursive=True):
    find_links.extend(['--find-links', d])
for d in glob.glob('/kaggle/input/**/packages', recursive=True):
    find_links.extend(['--find-links', d])
find_links_str = ' '.join(find_links)

# Install trl
if not find_spec('trl'):
    sh(f'pip install --no-index {find_links_str} trl', check=False)

# Install mamba_ssm + causal_conv1d
all_mamba = recursive_wheels('mamba_ssm-*.whl')
all_causal = recursive_wheels('causal*conv1d*.whl')
print(f'Found mamba wheels: {len(all_mamba)} | causal_conv1d wheels: {len(all_causal)}')

if not find_spec('causal_conv1d') and all_causal:
    w = pick_best(all_causal)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')

if not find_spec('mamba_ssm') and all_mamba:
    w = pick_best(all_mamba)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')

# Stub mamba3 modules
for _mod_name in [
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
]:
    _m = types.ModuleType(_mod_name)
    _m.__path__ = []
    _m.__package__ = _mod_name
    sys.modules[_mod_name] = _m
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

# Verify ALL required packages
import mamba_ssm
import datasets
import trl
import vllm  # built from source in cell 2
print(f'✅ mamba_ssm: {mamba_ssm.__version__}')
print(f'✅ datasets:  {datasets.__version__}')
print(f'✅ trl:       {trl.__version__}')
print(f'✅ vllm:      {vllm.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4: ALL IMPORTS
# ════════════════════════════════════════════════════════════════
import os, sys, gc, re, math, json, time, zipfile, warnings, shutil
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
import polars as pl
import kagglehub
from collections import defaultdict

from datasets import Dataset as HFDataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import GRPOTrainer, GRPOConfig

print(f'✅ All imports OK | PyTorch {torch.__version__} | '
      f'CUDA {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5: GRPO CONFIGURATION
# ════════════════════════════════════════════════════════════════

GRPO_LR = 5e-6
GRPO_NUM_GENERATIONS = 4
GRPO_MAX_COMPLETION = 2048
GRPO_TEMPERATURE = 1.0
GRPO_BETA = 0.0
GRPO_SUBSAMPLE = 10
GRPO_EPOCHS = 1
MAX_STEPS_GRPO = -1

LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
USE_RSLORA = False

OUTPUT_DIR = '/kaggle/working/sft_adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print(f'GRPO Config: lr={GRPO_LR}, gens={GRPO_NUM_GENERATIONS}, max_completion={GRPO_MAX_COMPLETION}')
print(f'  temperature={GRPO_TEMPERATURE}, beta={GRPO_BETA}, subsample={GRPO_SUBSAMPLE}')
print(f'  vLLM colocate mode enabled')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6: DATA LOADING — bit_manipulation ONLY
# ════════════════════════════════════════════════════════════════
import pandas as pd

# Find training CSV
TRAIN_CANDIDATES = [
    '/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv',
    '/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv',
]
TRAIN_PATH = None
for p in TRAIN_CANDIDATES:
    if os.path.exists(p):
        TRAIN_PATH = p
        break
if TRAIN_PATH is None:
    candidates = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    for c in candidates:
        if 'train' in os.path.basename(c).lower():
            TRAIN_PATH = c
            break
if TRAIN_PATH is None:
    raise FileNotFoundError(f'No training CSV found')
print(f'Using: {TRAIN_PATH}')

train_full = pl.read_csv(TRAIN_PATH)
print(f'Competition train data: {len(train_full)} samples')
print(f'Columns: {train_full.columns}')

def classify_puzzle(prompt):
    p = prompt.lower()
    if re.search(r'numeral system|base[- ]?\d|number.*convert|radix|secret number|roman', p):
        return 'Number Base Conversion'
    elif re.search(r'gravit|gravity|falling|free.?fall|acceleration due to', p):
        return 'Gravitational Constant'
    elif re.search(r'transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation', p):
        return 'Equation Transformation'
    elif re.search(r'encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text|decrypt', p):
        return 'Text Encryption'
    elif re.search(r'bit.?manipul|binary|8.?bit|bitwise|bit.*transform', p):
        return 'Bit Manipulation'
    elif re.search(r'unit.?conver|measurement|becomes.*\d|secret.*conver.*measur', p):
        return 'Unit Conversion'
    else:
        return 'Unknown'

train_full = train_full.with_columns(
    pl.col('prompt').map_elements(classify_puzzle, return_dtype=pl.Utf8).alias('puzzle_type')
)
print('\nPuzzle type distribution:')
print(train_full.group_by('puzzle_type').agg(pl.len().alias('count')).sort('count', descending=True))

# ── Filter to bit_manipulation ONLY ──
bit_df = train_full.filter(pl.col('puzzle_type') == 'Bit Manipulation')
print(f'\nBit Manipulation samples: {len(bit_df)}')

# ── Subsample if configured ──
if GRPO_SUBSAMPLE > 0 and GRPO_SUBSAMPLE < len(bit_df):
    bit_df = bit_df.sample(n=GRPO_SUBSAMPLE, seed=42)
    print(f'Subsampled to: {len(bit_df)}')

# ── Build GRPO dataset ──
BOXED_INSTRUCTION = PROMPT_SUFFIX

grpo_records = []
for row in bit_df.iter_rows(named=True):
    grpo_records.append({
        'prompt': [{'role': 'user', 'content': row['prompt'] + BOXED_INSTRUCTION}],
        'ground_truth': str(row['answer']),
    })

grpo_dataset = HFDataset.from_list(grpo_records)
print(f'GRPO dataset: {len(grpo_dataset)} prompts x {GRPO_NUM_GENERATIONS} gens')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7: MODEL LOADING + 0.72 SFT ADAPTER
# ════════════════════════════════════════════════════════════════

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print(f'Model path: {MODEL_PATH}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16,
)
print('✅ Base model loaded')

# ── Find and extract 0.72 SFT adapter ──
import glob as _glob

# Step 1: Search for adapter_config.json directly
adapter_candidates = _glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
if adapter_candidates:
    ADAPTER_PATH = os.path.dirname(adapter_candidates[0])
    print(f'Found adapter at: {ADAPTER_PATH}')
else:
    # Step 2: Look for submission.zip and extract it
    zip_candidates = _glob.glob('/kaggle/input/**/submission.zip', recursive=True)
    if not zip_candidates:
        zip_candidates = _glob.glob('/kaggle/input/**/*.zip', recursive=True)
    
    ADAPTER_PATH = '/kaggle/working/072_adapter'
    os.makedirs(ADAPTER_PATH, exist_ok=True)
    
    extracted = False
    for zp in zip_candidates:
        print(f'Trying zip: {zp}')
        try:
            import zipfile as _zf
            with _zf.ZipFile(zp) as z:
                names = z.namelist()
                if 'adapter_config.json' in names:
                    z.extractall(ADAPTER_PATH)
                    print(f'✅ Extracted adapter from {zp}: {names}')
                    extracted = True
                    break
        except Exception as e:
            print(f'  Skip: {e}')
    
    if not extracted:
        # Step 3: List everything available for debugging
        all_files = _glob.glob('/kaggle/input/**/*', recursive=True)
        print(f'All input files ({len(all_files)}):')
        for f in all_files[:50]:
            print(f'  {f}')
        raise FileNotFoundError('Cannot find adapter_config.json in any input')

print(f'Adapter path: {ADAPTER_PATH}')
print(f'Adapter files: {os.listdir(ADAPTER_PATH)}')

model = PeftModel.from_pretrained(model, ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()
print('✅ 0.72 SFT adapter loaded (trainable for GRPO)')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8: REWARD FUNCTIONS (proven 0.70 scheme)
# ════════════════════════════════════════════════════════════════

_reward_debug_counter = {"calls": 0}

def _normalize_answer(s):
    """Normalize answer for comparison with numeric tolerance."""
    s = s.strip()
    try:
        f = float(s)
        if f == int(f): return str(int(f))
        return str(f)
    except (ValueError, OverflowError):
        return s

def _answers_match(pred, gt, rel_tol=1e-2):
    """Match answers with numeric tolerance (competition uses 1e-2 relative)."""
    pred_n, gt_n = _normalize_answer(pred), _normalize_answer(gt)
    if pred_n == gt_n: return True
    if pred_n.lower() == gt_n.lower(): return True
    try:
        p, g = float(pred), float(gt)
        if g == 0: return abs(p) < 1e-6
        return abs(p - g) / abs(g) <= rel_tol
    except (ValueError, OverflowError):
        return False

def _extract_boxed(content):
    match = re.search(r'\\boxed\{([^}]*)\}', content, re.DOTALL)
    if match: return match.group(1).strip()
    match = re.search(r'boxed\{([^}]*)\}', content, re.DOTALL)
    if match: return match.group(1).strip()
    return None

def _get_content(completion):
    if isinstance(completion, list):
        return completion[-1]["content"] if completion else ""
    return completion

def cosine_reward(completions, ground_truth, **kwargs):
    """Cosine-scaled accuracy reward with numeric tolerance."""
    max_len = GRPO_MAX_COMPLETION
    rewards = []
    _reward_debug_counter["calls"] += 1
    show_debug = _reward_debug_counter["calls"] <= 2

    for i, (completion, gt) in enumerate(zip(completions, ground_truth)):
        content = _get_content(completion)
        extracted = _extract_boxed(content)
        clen = len(content)
        progress = min(clen / max(max_len, 1), 1.0)
        cos_scale = 0.5 * (1.0 + math.cos(math.pi * progress))

        if extracted is not None and _answers_match(extracted, gt):
            reward = 0.1 + 0.9 * cos_scale
        elif extracted is not None:
            reward = -0.1 - 0.9 * (1.0 - cos_scale)
        else:
            reward = -0.5 * progress

        rewards.append(reward)
        if show_debug and i < 2:
            print(f"  [COS] ext={extracted!r}, gt={gt!r}, r={reward:.3f}, len={clen}", flush=True)
    return rewards

def format_reward(completions, **kwargs):
    return [1.0 if _extract_boxed(_get_content(c)) is not None else 0.0 for c in completions]

def reasoning_quality_reward(completions, **kwargs):
    """Rewards structured reasoning (step markers, examples cited)."""
    rewards = []
    for c in completions:
        content = _get_content(c)
        score = 0.0
        if re.search(r'step\s*\d|example\s*\d|first|then|therefore|thus', content.lower()):
            score += 0.3
        if re.search(r'[=×÷+\-*/^]', content):
            score += 0.2
        boxed_pos = content.find('\\boxed')
        if boxed_pos > 100:
            score += 0.2
        words = content.split()
        if len(words) > 10:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio > 0.3:
                score += 0.3
        rewards.append(score)
    return rewards

# Sanity check
_reward_debug_counter["calls"] = 0
_test = cosine_reward(
    completions=["The answer is \\boxed{42}.", "No boxed.", "Wrong: \\boxed{99}." + " "*800],
    ground_truth=["42", "42", "42"],
)
print(f"Reward sanity: {[f'{r:.3f}' for r in _test]}  (expect ~[1.0, 0.0, ~-0.7])")
_reward_debug_counter["calls"] = 0

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11: TRAINING CALLBACKS
# ════════════════════════════════════════════════════════════════

class PrintLossCallback(TrainerCallback):
    def __init__(self, phase='SFT'):
        self.phase = phase
        self.start_time = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f'\n{"="*60}\n[{self.phase}] Training started — {state.max_steps} steps\n{"="*60}', flush=True)
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        elapsed = time.time() - self.start_time if self.start_time else 0
        loss = logs.get('loss', logs.get('train_loss'))
        lr = logs.get('learning_rate')
        parts = [f'[{self.phase}] step {state.global_step}/{state.max_steps}']
        if loss is not None: parts.append(f'loss={loss:.4f}')
        if lr is not None: parts.append(f'lr={lr:.2e}')
        parts.append(f'elapsed={elapsed/60:.1f}min')
        print(' | '.join(parts), flush=True)
    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time if self.start_time else 0
        print(f'\n[{self.phase}] Complete — {state.global_step} steps in {elapsed/60:.1f}min', flush=True)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10: GRPO TRAINING with vLLM colocate
# ════════════════════════════════════════════════════════════════

grpo_kwargs = dict(
    output_dir=OUTPUT_DIR + '_grpo',
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=GRPO_LR,
    lr_scheduler_type='cosine',
    warmup_steps=5,
    weight_decay=0.0,
    num_generations=GRPO_NUM_GENERATIONS,
    generation_batch_size=GRPO_NUM_GENERATIONS,
    max_completion_length=GRPO_MAX_COMPLETION,
    temperature=GRPO_TEMPERATURE,
    beta=GRPO_BETA,
    logging_steps=1,
    save_strategy='no',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    max_grad_norm=0.1,
    remove_unused_columns=False,
    # vLLM colocate
    use_vllm=True,
    vllm_mode='colocate',
    vllm_gpu_memory_utilization=0.4,
)
if MAX_STEPS_GRPO > 0:
    grpo_kwargs['max_steps'] = MAX_STEPS_GRPO

grpo_config = GRPOConfig(**grpo_kwargs)

tokenizer.padding_side = 'left'

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    reward_funcs=[
        cosine_reward,
        format_reward,
        reasoning_quality_reward,
    ],
    callbacks=[PrintLossCallback('GRPO')],
)

print(f'Starting GRPO training with vLLM colocate...')
print(f'  Dataset: {len(grpo_dataset)} prompts x {GRPO_NUM_GENERATIONS} gens')
print(f'  LR={GRPO_LR}, beta={GRPO_BETA}, temp={GRPO_TEMPERATURE}')
t0 = time.time()
grpo_trainer.train()
elapsed = time.time() - t0
print(f'GRPO done in {elapsed/60:.1f} min')

model.save_pretrained(OUTPUT_DIR)
del grpo_trainer; gc.collect(); torch.cuda.empty_cache()
print('GRPO complete!')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 14: PACKAGE SUBMISSION
# ════════════════════════════════════════════════════════════════

SUBMISSION_DIR = '/kaggle/working/submission_adapter'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Copy adapter files
required_files = ['adapter_config.json', 'adapter_model.safetensors']
for fname in required_files:
    src = os.path.join(OUTPUT_DIR, fname)
    dst = os.path.join(SUBMISSION_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  {fname}: {os.path.getsize(dst)/1024/1024:.1f} MB')
    else:
        print(f'  ⚠️ {fname} not found!')

# Fix adapter_config for submission
config_path = os.path.join(SUBMISSION_DIR, 'adapter_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = json.load(f)
    cfg['base_model_name_or_path'] = BASE_MODEL_NAME
    cfg['inference_mode'] = True
    cfg['lora_dropout'] = 0.0
    with open(config_path, 'w') as f:
        json.dump(cfg, f, indent=2)

# Create submission.zip
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)
            print(f'  Added {fname}')

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f'\n✅ submission.zip: {zip_sz:.1f} MB')
print('Done! Ready to submit.')
